# Scanorama default - human_pbmc_sequencing_techs

Seeds: `0-9`

Metrics: BAMI, CASW, SCGRAPHM, ISOLISI, BRAS, KBET, CPCR

This notebook is intentionally unexecuted.


In [1]:
import gc
import os
import sys

os.environ["PYTHONNOUSERSITE"] = "1"
sys.path = [
    path
    for path in sys.path
    if ".local/lib/python3.10/site-packages" not in path
]

PROJECT_ROOT = (
    "/ubc/cs/research/beaver/projects/"
    "Single-cell-integration-metrics"
)
os.environ["R_HOME"] = (
    f"{PROJECT_ROOT}/CondaEnv/harmony_optuna/lib/R"
)
sys.path.insert(0, f"{PROJECT_ROOT}/scib/")

import anndata as ad
import numpy as np
import pandas as pd
import scanorama
import scanpy as sc
import scipy.sparse as sp

from scib.metrics.metrics import metrics


In [2]:
DATASET_NAME = 'human_pbmc_sequencing_techs'
DATA_PATH = f"{PROJECT_ROOT}/data/{DATASET_NAME}.h5ad"
BATCH_KEY = 'Method'
LABEL_KEY = 'CellType'

SELECTED_BIO_METRICS = ('BAMI', 'CASW', 'SCGRAPHM', 'ISOLISI')
SELECTED_BATCH_METRICS = ('BRAS', 'KBET', 'CPCR')
SELECTED_METRICS = SELECTED_BIO_METRICS + SELECTED_BATCH_METRICS

N_SEEDS = 10
N_NEIGHBORS = 15
N_TOP_HVG = 2000
INTERNAL_DIMRED = 50
BATCH_SIZE = 5000
APPROX = True
UNION = False

EMBED_KEY = "X_scanorama"
CLUSTER_KEY = "leiden_opt"
METRIC_BATCH_KEY = "scanorama_metric_batch"
METRIC_LABEL_KEY = "scanorama_metric_label"

assert DATASET_NAME == 'human_pbmc_sequencing_techs'
assert BATCH_KEY == 'Method'
assert LABEL_KEY == 'CellType'
assert SELECTED_METRICS == ('BAMI', 'CASW', 'SCGRAPHM', 'ISOLISI', 'BRAS', 'KBET', 'CPCR')

pd.Series(
    {
        "dataset": DATASET_NAME,
        "data_path": DATA_PATH,
        "batch_key": BATCH_KEY,
        "label_key": LABEL_KEY,
        "metrics": list(SELECTED_METRICS),
        "seeds": list(range(N_SEEDS)),
        "internal_dimred": INTERNAL_DIMRED,
        "batch_size": BATCH_SIZE,
        "n_top_hvg": N_TOP_HVG,
    },
    name="configuration",
)


dataset                                  human_pbmc_sequencing_techs
data_path          /ubc/cs/research/beaver/projects/Single-cell-i...
batch_key                                                     Method
label_key                                                   CellType
metrics            [BAMI, CASW, SCGRAPHM, ISOLISI, BRAS, KBET, CPCR]
seeds                                 [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]
internal_dimred                                                   50
batch_size                                                      5000
n_top_hvg                                                       2000
Name: configuration, dtype: object

In [3]:
def validate_dataset_contents():
    adata_backed = ad.read_h5ad(DATA_PATH, backed="r")
    try:
        required_keys = (BATCH_KEY, LABEL_KEY)
        missing_keys = [
            key for key in required_keys if key not in adata_backed.obs
        ]
        if missing_keys:
            raise ValueError(
                f"Missing obs keys {missing_keys}. "
                f"Available keys: {list(adata_backed.obs.columns)}"
            )

        obs = adata_backed.obs[[BATCH_KEY, LABEL_KEY]].copy()
        pseudotime = None
        if "TRAJ" in SELECTED_METRICS:
            if "dpt_pseudotime" not in adata_backed.obs:
                raise ValueError(
                    "TRAJ requires adata.obs['dpt_pseudotime']."
                )
            pseudotime = adata_backed.obs["dpt_pseudotime"].copy()

        n_obs = int(adata_backed.n_obs)
        n_vars = int(adata_backed.n_vars)
    finally:
        adata_backed.file.close()

    if obs.isna().any().any():
        raise ValueError(
            "Missing batch/label values: "
            f"{obs.isna().sum().to_dict()}"
        )

    n_batches = int(obs[BATCH_KEY].nunique())
    n_labels = int(obs[LABEL_KEY].nunique())
    if n_batches < 2:
        raise ValueError(
            f"Batch metrics require at least 2 batches; found {n_batches}."
        )

    batch_coverage = obs.groupby(
        LABEL_KEY,
        observed=True,
    )[BATCH_KEY].nunique()
    label_counts = obs.groupby(LABEL_KEY, observed=True).size()

    if "TRAJ" in SELECTED_METRICS:
        finite_pseudotime = pseudotime.dropna()
        if (
            len(finite_pseudotime) < 2
            or finite_pseudotime.nunique() < 2
        ):
            raise ValueError(
                "TRAJ requires at least 2 usable pseudotime values."
            )

    if "ISOLISI" in SELECTED_METRICS:
        min_batch_coverage = int(batch_coverage.min())
        if min_batch_coverage >= n_batches:
            raise ValueError(
                "ISOLISI is enabled, but every label appears in all batches."
            )

    eligible_cpcr_labels = [
        str(label)
        for label in label_counts.index
        if int(label_counts.loc[label]) >= 100
        and int(batch_coverage.loc[label]) >= 2
    ]
    if "CPCR" in SELECTED_METRICS and not eligible_cpcr_labels:
        raise ValueError(
            "CPCR requires a label with at least 100 cells across "
            "at least 2 batches."
        )

    return {
        "n_obs": n_obs,
        "n_vars": n_vars,
        "n_batches": n_batches,
        "n_labels": n_labels,
        "cpcr_eligible_labels": len(eligible_cpcr_labels),
    }


validate_dataset_contents()


{'n_obs': 30975,
 'n_vars': 33694,
 'n_batches': 9,
 'n_labels': 9,
 'cpcr_eligible_labels': 9}

In [4]:
SCANORAMA_PARAMS = {'knn': 20, 'sigma': 15.0, 'alpha': 0.1}
assert tuple(SCANORAMA_PARAMS) == ("knn", "sigma", "alpha")

pd.Series(
    {
        **SCANORAMA_PARAMS,
        "batch_size": BATCH_SIZE,
        "approx": APPROX,
        "union": UNION,
        "internal_dimred": INTERNAL_DIMRED,
        "n_top_hvg": N_TOP_HVG,
    },
    name="default_setting",
)

def check_keys(adata):
    missing_keys = [
        key
        for key in (BATCH_KEY, LABEL_KEY)
        if key not in adata.obs
    ]
    if missing_keys:
        raise ValueError(
            f"Missing keys in adata.obs: {missing_keys}. "
            f"Available keys: {list(adata.obs.columns)}"
        )


def ensure_csr(matrix):
    if sp.issparse(matrix):
        return matrix.tocsr()
    return sp.csr_matrix(matrix)


def split_by_batch(adata):
    batches = adata.obs[BATCH_KEY].astype("category")
    batch_names = list(batches.cat.categories)
    parts = []
    index_parts = []

    for batch_name in batch_names:
        indices = np.where(batches.values == batch_name)[0]
        index_parts.append(indices)
        parts.append(adata[indices, :].copy())

    return parts, batch_names, index_parts


def hvg_subset(adata):
    if "highly_variable" not in adata.var:
        sc.pp.highly_variable_genes(
            adata,
            flavor="seurat",
            n_top_genes=N_TOP_HVG,
        )

    hvg_mask = (
        adata.var["highly_variable"]
        .fillna(False)
        .astype(bool)
        .to_numpy()
    )
    if int(hvg_mask.sum()) == 0:
        raise ValueError("No highly variable genes selected.")
    return adata[:, hvg_mask].copy()


def reassemble_scanorama_embedding(parts, index_parts, n_obs):
    embedding_concat = np.concatenate(
        [part.obsm[EMBED_KEY] for part in parts],
        axis=0,
    )
    concat_indices = np.concatenate(index_parts)
    inverse = np.empty(n_obs, dtype=np.int64)
    inverse[concat_indices] = np.arange(n_obs, dtype=np.int64)
    return embedding_concat[inverse, :]


def run_scanorama_setting(adata, seed):
    np.random.seed(seed)
    check_keys(adata)

    adata_work = hvg_subset(adata.copy())
    adata_work.X = ensure_csr(adata_work.X)
    parts, batch_names, index_parts = split_by_batch(adata_work)

    scanorama.integrate_scanpy(
        parts,
        batch_size=BATCH_SIZE,
        verbose=0,
        ds_names=batch_names,
        dimred=INTERNAL_DIMRED,
        approx=APPROX,
        sigma=float(SCANORAMA_PARAMS["sigma"]),
        alpha=float(SCANORAMA_PARAMS["alpha"]),
        knn=int(SCANORAMA_PARAMS["knn"]),
        union=UNION,
        hvg=None,
        seed=int(seed),
    )

    embedding = reassemble_scanorama_embedding(
        parts,
        index_parts,
        n_obs=adata_work.n_obs,
    )
    if embedding.shape[0] != adata_work.n_obs:
        raise ValueError(
            f"Scanorama output shape {embedding.shape} is incompatible "
            f"with n_obs={adata_work.n_obs}."
        )
    if not np.all(np.isfinite(embedding)):
        raise ValueError(
            "Scanorama produced NaN or Inf in X_scanorama."
        )

    adata_int = adata_work.copy()
    adata_int.obsm[EMBED_KEY] = np.asarray(
        embedding,
        dtype=np.float32,
    )
    return adata_int


In [5]:
def metric_value(metric_series, key):
    if key not in metric_series.index:
        raise KeyError(
            f"Metric {key} not found. "
            f"Available metrics: {list(metric_series.index)}"
        )

    value = metric_series[key]
    if isinstance(value, (list, tuple, np.ndarray)):
        values = np.asarray(value, dtype=float)
        if values.size == 0 or not np.all(np.isfinite(values)):
            raise ValueError(
                f"Metric {key} returned non-finite values: {values}"
            )
        return float(np.mean(values))

    value = float(value)
    if not np.isfinite(value):
        raise ValueError(
            f"Metric {key} returned a non-finite value: {value}"
        )
    return value


def compute_joint_metrics(adata_raw, adata_int):
    sc.pp.neighbors(
        adata_int,
        use_rep=EMBED_KEY,
        n_neighbors=N_NEIGHBORS,
        random_state=0,
    )

    metric_key_map = {
        METRIC_BATCH_KEY: BATCH_KEY,
        METRIC_LABEL_KEY: LABEL_KEY,
    }
    for adata_name, adata_obj in (
        ("adata_raw", adata_raw),
        ("adata_int", adata_int),
    ):
        existing_aliases = [
            alias for alias in metric_key_map if alias in adata_obj.obs
        ]
        if existing_aliases:
            raise ValueError(
                f"Reserved metric obs keys already exist in {adata_name}: "
                f"{existing_aliases}"
            )
        for alias, source_key in metric_key_map.items():
            adata_obj.obs[alias] = adata_obj.obs[source_key].copy()

    metric_df = metrics(
        adata=adata_raw,
        adata_int=adata_int,
        batch_key=METRIC_BATCH_KEY,
        label_key=METRIC_LABEL_KEY,
        embed=EMBED_KEY,
        cluster_key=CLUSTER_KEY,
        verbose=False,
        bami_="BAMI" in SELECTED_METRICS,
        casw_="CASW" in SELECTED_METRICS,
        scgraphm_="SCGRAPHM" in SELECTED_METRICS,
        traj_="TRAJ" in SELECTED_METRICS,
        isolisi_="ISOLISI" in SELECTED_METRICS,
        bras_="BRAS" in SELECTED_METRICS,
        kbet_="KBET" in SELECTED_METRICS,
        cpcr_="CPCR" in SELECTED_METRICS,
    )
    metric_series = metric_df.iloc[:, 0]

    result = {
        key: metric_value(metric_series, key)
        for key in SELECTED_METRICS
    }
    bio_score = float(
        np.mean([result[key] for key in SELECTED_BIO_METRICS])
    )
    batch_score = float(
        np.mean([result[key] for key in SELECTED_BATCH_METRICS])
    )
    joint_score = float(0.6 * bio_score + 0.4 * batch_score)

    if not np.all(np.isfinite([bio_score, batch_score, joint_score])):
        raise ValueError(
            "Aggregation produced a non-finite score: "
            f"bio={bio_score}, batch={batch_score}, "
            f"joint={joint_score}"
        )

    result["bio_score"] = bio_score
    result["batch_score"] = batch_score
    result["joint_score"] = joint_score
    return result


In [6]:
results = []

for seed in range(N_SEEDS):
    print(f"\n===== seed {seed} =====", flush=True)

    adata_raw = ad.read_h5ad(DATA_PATH)
    adata_int = run_scanorama_setting(adata_raw, seed=seed)
    metric_result = compute_joint_metrics(adata_raw, adata_int)

    row = {"seed": seed, **metric_result}
    results.append(row)
    print(pd.Series(row), flush=True)

    del adata_raw, adata_int
    gc.collect()

results_df = pd.DataFrame(results)
results_df



===== seed 0 =====


After 10x Chromium (v2): 4999 genes
After 10x Chromium (v2) A: 4999 genes
After 10x Chromium (v2) B: 4999 genes
After 10x Chromium (v3): 4999 genes
After CEL-Seq2: 4999 genes
After Drop-seq: 4999 genes
After Seq-Well: 4999 genes
After Smart-seq2: 4999 genes
After inDrops: 4999 genes
Found 4999 genes among all datasets


Cell type silhouette score...


Per cell type PC regression...


/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/harmony_optuna/lib/python3.10/site-packages/scanpy/preprocessing/_pca/__init__.py:226: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(
/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/harmony_optuna/lib/python3.10/site-packages/scanpy/preprocessing/_pca/__init__.py:226: FutureWarning: Argument `use_highly_variable` is deprecated, consider using the mask argument. Use_highly_variable=True can be called through mask_var="highly_variable". Use_highly_variable=False can be called through mask_var=None
  mask_var_param, mask_var = _handle_mask_var(
/ubc/cs/research/beaver/projects/Single-cell-integration-metrics/CondaEnv/harmony_optuna/lib/python3.10/site-packages/sc

...

Cluster for leiden_opt_2.0 with leiden


resolution: 2.0, bAMI: 0.6050537654952988


optimised clustering against scanorama_metric_label
optimal cluster resolution: 1.0
optimal score: 0.6252554802491465
BRAS...


seed           9.000000
BAMI           0.625255
CASW           0.531271
SCGRAPHM       0.882923
ISOLISI        0.862993
BRAS           0.773932
KBET           0.078131
CPCR           0.606839
bio_score      0.725611
batch_score    0.486301
joint_score    0.629887
dtype: float64


,seed,BAMI,CASW,SCGRAPHM,ISOLISI,BRAS,KBET,CPCR,bio_score,batch_score,joint_score
0,0,0.652803,0.529428,0.878126,0.869203,0.774579,0.085577,0.616472,0.732390,0.492209,0.636318
1,1,0.628535,0.529592,0.870798,0.891606,0.775650,0.087388,0.610041,0.730133,0.491026,0.634490
2,2,0.661207,0.530541,0.870211,0.915721,0.777029,0.088410,0.614175,0.744420,0.493205,0.643934
3,3,0.630512,0.530519,0.877114,0.913997,0.776014,0.080123,0.611858,0.738035,0.489332,0.638554
4,4,0.642659,0.529168,0.875708,0.930582,0.774742,0.081205,0.602434,0.744529,0.486127,0.641168
5,5,0.624447,0.531665,0.882612,0.932744,0.776348,0.081034,0.616731,0.742867,0.491371,0.642269
6,6,0.645978,0.530117,0.870095,0.852171,0.779631,0.087256,0.621829,0.724590,0.496239,0.633250
7,7,0.626494,0.531457,0.872290,0.886492,0.774331,0.088056,0.608084,0.729184,0.490157,0.633573
8,8,0.637661,0.530752,0.884904,0.931788,0.773448,0.079103,0.606026,0.746276,0.486193,0.642243
9,9,0.625255,0.531271,0.882923,0.862993,0.773932,0.078131,0.606839,0.725611,0.486301,0.629887


In [7]:
score_columns = [
    *SELECTED_METRICS,
    "bio_score",
    "batch_score",
    "joint_score",
]

summary_df = pd.DataFrame(
    {
        "mean": results_df[score_columns].mean(),
        "std": results_df[score_columns].std(ddof=0),
        "min": results_df[score_columns].min(),
        "max": results_df[score_columns].max(),
    }
)
summary_df


,mean,std,min,max
BAMI,0.637555,0.012105,0.624447,0.661207
CASW,0.530451,0.000825,0.529168,0.531665
SCGRAPHM,0.876478,0.005326,0.870095,0.884904
ISOLISI,0.898730,0.028815,0.852171,0.932744
BRAS,0.775570,0.001727,0.773448,0.779631
KBET,0.083628,0.003862,0.078131,0.088410
CPCR,0.611449,0.005601,0.602434,0.621829
bio_score,0.735803,0.007951,0.724590,0.746276
batch_score,0.490216,0.003165,0.486127,0.496239
joint_score,0.637568,0.004506,0.629887,0.643934
